In [4]:
# 🔧 Instalación de librerías necesarias (descomenta si es necesario en Colab)
#!pip install nltk scikit-learn

# 📚 Importar librerías
import nltk
import random
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 📥 Descargar recursos de NLTK
nltk.download('punkt')
nltk.download('stopwords')

# 🧠 Base de conocimiento para Papelería Virtual en Cartago: Preguntas de Servicio Inteligente
base_conocimiento = {
    "logistica_envios": {
        "¿Hay algún recargo por pagar contra entrega en Cartago?": "No, el *pago contra entrega* en efectivo no tiene *recargo* adicional. Solo aplica el costo base del domicilio.",
        "Si pido hoy antes de las 3 PM, ¿cuándo llega el pedido?": "Si pides y pagas *antes de las 3:00 p.m., garantizamos la entrega el **mismo día* antes de las 6:00 p.m. dentro de la zona urbana.",
        "¿Hacen envíos a dos direcciones diferentes en el mismo pedido?": "Sí, pero se cobraría el *costo de domicilio por separado* para cada dirección de entrega. Te recomendamos hacer dos pedidos distintos.",
        "Necesito 100 resmas de papel para una empresa, ¿cuánto se demoran en despachar?": "Para pedidos grandes (más de 50 unidades), el tiempo de despacho es de *24 a 48 horas*, ya que requieren logística especial y revisión de stock en bodega.",
        "¿Qué pasa si no estoy en casa cuando llegue el domiciliario?": "El domiciliario te llamará para coordinar. Si no responde, esperará 5 minutos o intentará *reprogramar la entrega* para el siguiente día, lo cual podría generar un *cargo extra por el reenvío*."
    },

    "promociones_ventas": {
        "¿Tienen algún descuento especial para estudiantes o profesores?": "Sí, tenemos un *descuento fijo del 5%* en útiles escolares presentando un carné estudiantil o certificación docente vigente al momento de pagar.",
        "¿Hay promociones por la compra de útiles de la marca Faber-Castell?": "¡Claro! Generalmente ofrecemos un *3x2 en lápices y colores Faber-Castell* al inicio de cada mes. Revisa la sección 'Ofertas del Mes'.",
        "¿Qué debo hacer si quiero comprar al por mayor para una tienda pequeña?": "Debes contactar a nuestra línea de *Ventas Corporativas* (300 987 6543) para acceder a precios de mayorista con descuentos superiores al 20%.",
        "¿Tienen alguna opción de crédito o 'paga después' para empresas?": "Sí, ofrecemos un plan de crédito a 15 días a clientes corporativos con un mínimo de 3 pedidos previos y previa aprobación de nuestra área financiera.",
        "Si compro un combo de colores, ¿puedo cambiar un artículo por otro similar?": "Lamentablemente, los *combos* tienen precios fijos y no son modificables. Debes comprar los artículos por separado si deseas personalizarlos."
    },

    "garantias_soporte": {
        "Compré un marcador permanente que llegó seco, ¿qué hago?": "¡Lamentamos el inconveniente! Tienes *3 días* para reportar defectos de fábrica. Te enviaremos un *reemplazo gratuito* en tu próximo pedido o con un domiciliario especial si es urgente.",
        "¿Cuál es el proceso para pedir una devolución de dinero?": "Si el producto está sellado, inicia el proceso contactando a soporte. La devolución se realiza a la misma cuenta de origen en un plazo de *3 a 5 días hábiles*.",
        "¿Cómo sé si mis datos personales y de pago están seguros en la web?": "Nuestra plataforma usa encriptación *SSL/TLS de 256 bits* y no almacenamos información sensible de tarjetas. Tu seguridad es nuestra prioridad.",
        "La página me da un error al finalizar la compra, ¿a quién contacto?": "Si tienes problemas técnicos, por favor envía una captura de pantalla al WhatsApp de *Soporte Técnico* (300 543 2109) para ayudarte a completar el pedido manualmente.",
        "¿Los precios del sitio web ya incluyen el IVA?": "Sí, todos los precios mostrados en nuestra papelería virtual *incluyen el IVA* (Impuesto al Valor Agregado) correspondiente a cada producto."
    }
}

# 🤝 Respuestas rápidas para Papelería Virtual
respuestas_saludo = [
    "¡Hola! 👋 Soy Asistente Papelera. Dime: ¿Buscas un útil específico o necesitas información sobre el *domicilio en Cartago*?",
    "¡Bienvenido/a a la Papelería Virtual! 😊 ¿En qué puedo ayudarte hoy con tu pedido o consulta?",
    "¡Buenos días/tardes! 🌟 ¿Cómo puedo hacer tu *compra más fácil*? Pregúntame sobre envíos, stock o promociones.",
    "¡Hola desde Cartago! 🚀 ¿Necesitas saber el costo de envío o tienes alguna duda sobre un producto?"
]

respuestas_despedida = [
    "¡Gracias por visitarnos! 👋 Espero verte pronto. ¡Hacemos envíos entre semana hasta las 8 PM, y sabados hasta las 2 PM!",
    "¡Hasta pronto! 😊 Tu pedido está a solo un clic. ¡Que tengas un gran día!",
    "¡Adiós! 🌟 No olvides preguntar por el *domicilio gratis* en compras mayores a $50.000.",
    "¡Finaliza tu compra cuando quieras! 🛒 Si necesitas algo más, solo escribe 'Hola'."
]

# 🧹 Funciones de procesamiento de texto
def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s¿?áéíóúñ]', ' ', texto)
    return ' '.join(texto.split())

def tokenizar_texto(texto):
    try:
        return nltk.word_tokenize(texto, language='spanish')
    except:
        return texto.split()

# 🎯 Funciones del chatbot
def detectar_saludo(pregunta):
    saludos = ['hola','holaa', 'buenos días', 'buenas tardes', 'buenas noches', 'saludos', 'qué tal', 'hey']
    return any(saludo in limpiar_texto(pregunta) for saludo in saludos)

def detectar_despedida(pregunta):
    despedidas = ['adiós', 'hasta luego', 'nos vemos', 'chau', 'chao', 'bye', 'salir', 'terminar']
    return any(despedida in limpiar_texto(pregunta) for despedida in despedidas)

def calcular_similitud(pregunta_usuario, pregunta_base):
    try:
        vectorizador = CountVectorizer().fit([pregunta_usuario, pregunta_base])
        vectores = vectorizador.transform([pregunta_usuario, pregunta_base])
        return cosine_similarity(vectores[0], vectores[1])[0][0]
    except:
        palabras_usuario = set(tokenizar_texto(pregunta_usuario))
        palabras_base = set(tokenizar_texto(pregunta_base))
        return len(palabras_usuario & palabras_base) / max(len(palabras_usuario), len(palabras_base))

def encontrar_mejor_respuesta(pregunta_usuario):
    if detectar_saludo(pregunta_usuario):
        return random.choice(respuestas_saludo)
    if detectar_despedida(pregunta_usuario):
        return random.choice(respuestas_despedida)

    mejor_respuesta = ""
    mejor_puntuacion = 0
    materia_encontrada = ""

    for materia, preguntas in base_conocimiento.items():
        for pregunta_base, respuesta in preguntas.items():
            puntuacion = calcular_similitud(limpiar_texto(pregunta_usuario), limpiar_texto(pregunta_base))
            if puntuacion > mejor_puntuacion:
                mejor_puntuacion = puntuacion
                mejor_respuesta = respuesta
                materia_encontrada = materia
    
    if mejor_puntuacion > 0.2:
        return f"📚 [{materia_encontrada.title()}] {mejor_respuesta}"
    else:
        return random.choice([
            "🤔 No estoy seguro, ¿puedes reformular la pregunta?",
            "🔍 Intenta preguntarme sobre matemáticas, ciencias, historia o literatura.",
            "❓ Hmm, no tengo esa información."
        ])

    # 💬 Interfaz principal
def iniciar_chatbot():
    print("🤖 ChatBot Escolar v1.0")
    print("Escribe 'salir' para terminar la conversación.\n")

    while True:
        usuario_input = input("🙋 Tú: ").strip()
        if not usuario_input:
            print("🤖 Bot: Por favor, escribe algo.")
            continue
        if detectar_despedida(usuario_input):
            print(f"🤖 Bot: {random.choice(respuestas_despedida)}")
            break
        respuesta = encontrar_mejor_respuesta(usuario_input)
        print(f"🤖 Bot: {respuesta}")

# 🚀 Iniciar chatbot
iniciar_chatbot()

ModuleNotFoundError: No module named 'nltk'